In [1]:
from spiral import Spiral
import pyarrow as pa

sp = Spiral()
project = sp.project("external-805943")

# Random IDs are discouraged; use monotonically increasing IDs where possible, like UUIDv7.
table = project.create_table("experiments-v1", key_schema=pa.schema({"data_key": pa.string()}), exist_ok=True)

In [2]:
import numpy as np
from pathlib import Path
import yaml
import json

# Loads experiment data from a directory. Download experiment referenced in `inspect_data.ipynb`.
experiment_data_path = Path("35_3832003544063_1_V1")

with open(experiment_data_path / "meta.json", "r") as f:
    global_meta = json.load(f)

In [3]:
eye_tracker_means = np.load(experiment_data_path / "eye_tracker" / "meta" / "means.npy")
eye_tracker_stds = np.load(experiment_data_path / "eye_tracker" / "meta" / "stds.npy")
with open(experiment_data_path / "eye_tracker" / "meta.yml", "r") as f:
    eye_tracker_meta = yaml.safe_load(f)

eye_tracker_data = np.memmap(experiment_data_path / "eye_tracker" / "data.mem",
                             dtype=eye_tracker_meta["dtype"],
                             mode="r",
                             shape=(eye_tracker_meta["n_timestamps"], eye_tracker_meta["n_signals"]),)

In [4]:
responses_means = np.load(experiment_data_path / "responses_30Hz_no_filtering" / "meta" / "means.npy")
responses_stds = np.load(experiment_data_path / "responses_30Hz_no_filtering" / "meta" / "stds.npy")
with open(experiment_data_path / "responses_30Hz_no_filtering" / "meta.yml", "r") as f:
    responses_meta = yaml.safe_load(f)

responses_data = np.memmap(experiment_data_path / "responses_30Hz_no_filtering" / "data.mem",
                           dtype=responses_meta["dtype"],
                           mode="r",
                           shape=(responses_meta["n_timestamps"], responses_meta["n_signals"]),)

In [9]:
screen_timestamps = np.load(experiment_data_path / "screen" / "timestamps.npy")
with open(experiment_data_path / "screen" / "meta.yml", "r") as f:
    screen_meta = yaml.safe_load(f)

# FIXME(marko): Figure out cell sizing.
#   ArrowCapacityError: array cannot contain more than 2147483646 bytes, have 2152886400
screen_limit = 50

screen_data_meta = []
for i, frame_meta_path in enumerate(sorted((experiment_data_path / "screen" / "meta").glob("*.yml"))):
    # NOTE(marko): data starts from 1, so skip 0 metadata
    if i == 0:
        continue
    if i == screen_limit + 1:
        break
    with open(frame_meta_path, "r") as f:
        screen_data_meta.append(yaml.safe_load(f))

print(f"Loaded metadata for {len(screen_data_meta)} frames")

screen_data = []
for i, frame_path in enumerate(sorted((experiment_data_path / "screen" / "data").glob("*.npy"))):
    if i == screen_limit:
        break
    # Store frames as bytes
    screen_data.append(np.load(frame_path).tobytes(order='C'))

print(f"Loaded data for {len(screen_data)} frames")

Loaded metadata for 50 frames
Loaded data for 50 frames


In [13]:
experiment = dict(global_meta)
experiment["eye_tracker"] = dict(eye_tracker_meta)
experiment["eye_tracker"]["means"] = eye_tracker_means
experiment["eye_tracker"]["stds"] = eye_tracker_stds
experiment["eye_tracker"]["data"] = {
    "x": eye_tracker_data[:, 0],
    "y": eye_tracker_data[:, 1],
    "pupil": eye_tracker_data[:, 2],
}
experiment["responses"] = dict(responses_meta)
experiment["responses"]["means"] = responses_means
experiment["responses"]["stds"] = responses_stds
experiment["responses"]["data"] = {
    "signals": responses_data.tolist()
}
experiment["screen"] = dict(screen_meta)
experiment["screen"]["timestamps"] = screen_timestamps
experiment["screen"]["metas"] = screen_data_meta
experiment["screen"]["data"] = {
    "frames": screen_data
}

-- is_valid: all not null
-- child 0 type: struct<database: struct<host: string, password: string, schema_name: string, user:  (... 1387 chars omitted)  -- is_valid: all not null
  -- child 0 type: struct<host: string, password: string, schema_name: string, user: string>
    -- is_valid: all not null
    -- child 0 type: string
      [
        "at-database3.stanford.edu"
      ]
    -- child 1 type: string
      [
        ""
      ]
    -- child 2 type: string
      [
        "enigma_acq"
      ]
    -- child 3 type: string
      [
        ""
      ]
  -- child 1 type: struct<compute_report: bool, default_fs: int64, export_suffix: string, output_base_ (... 280 chars omitted)    -- is_valid: all not null
    -- child 0 type: bool
      [
        true
      ]
    -- child 1 type: int64
      [
        30000
      ]
    -- child 2 type: string
      [
        "permissive_stability_criteria"
      ]
    -- child 3 type: string
      [
        "/mnt/stor02/enigma/modeling_pipeline/export/go

In [14]:
table.write([experiment])

2026-01-12T11:12:37.106682Z  INFO transaction.commit: spiral_table::transaction: Transaction committed successfully table_id=table_9br9dn operation_count=35 retry_attempt=0


In [15]:
table.schema()

Schema({data_key=utf8?, config={database={host=utf8?, password=utf8?, schema_name=utf8?, user=utf8?}?, export={compute_report=bool?, default_fs=i64?, export_suffix=utf8?, output_base_dir=utf8?, overwrite=bool?, target_sampling_rate=i64?, use_functional_criteria=bool?, use_regularity_criteria=bool?, use_stability_criteria=bool?, quality_thresholds={good_spikes_fraction=f64?, loo_slope=f64?, presence_ratio=f64?, variance_explained=f64?}?}?, gaze={behavior_mnt_dir=list(utf8?)?, blink_acc_threshold=f64?, blink_detection=bool?}?, processing={n_jobs=i64?}?, screen={dtype=utf8?, frame_t_max=f64?, frame_t_min=f64?, frames_in_split=i64?, gray_value=f64?, max_delta_pauses=f64?, new_h=i64?, new_w=i64?, original_video_h=i64?, post_stim_gray_s=f64?, pre_stim_gray_s=f64?, save_as_normalized=bool?, normalization={means=list(f64?)?, stds=list(f64?)?}?}?, session={auto_discover=bool?, beh_offset=i64?, beh_path=utf8?, brain_area=utf8?, electrode_id=i64?, export_gaze=bool?, export_screen=bool?, name=utf8